In [1]:
#Imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
import pandas as pd
import numpy as np
import os

In [2]:
#Uploaded Data
data_path = '/kaggle/input/datasets/mframos2/competition-training-and-test-csv'

train_df = pd.read_csv(os.path.join(data_path, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_path, 'test.csv'))
sample_submission = pd.read_csv(os.path.join(data_path, 'sample_submission.csv'))

In [3]:
#Initializing the vectorizer
class TextToFeatures:
    def __init__(self):
        """
        Initializes an object for converting texts to features.    
        """        
        self.vectorizer = CountVectorizer(
            ngram_range =(1,2),
            max_features=5000,
            min_df=2
        )
        self.is_fitted = False

    def fit(self, training_texts):
   
        texts_list = list(training_texts)
        self.vectorizer.fit(texts_list)
        self.is_fitted = True

    def transform(self, texts):
        """Creates the matrix"""
        texts_list = list(texts)
        feature_matrix = self.vectorizer.transform(texts_list)
        return feature_matrix

#Creating the index
class TextToLabels:
    def __init__(self):
        """
        Initializes an object for converting texts to labels.    
        """
        self.encoder = LabelEncoder()
        self.is_fitted = False

    def fit(self, training_labels):
        """Assign labels to each integer"""
        labels_list = list(training_labels)
        self.encoder.fit(labels_list)
        self.is_fitted = True

    def transform(self, labels):
        """Label vector from a sequence of labels"""
        labels_list = list(labels)
        encoded = self.encoder.transform(labels_list)
        return np.array(encoded)

    def index(self, label):
        """Index of the vocabulary"""
        l2i = {}
        for i, lbl in enumerate(self.encoder.classes_):
            l2i[lbl] = i
        return l2i.get(label)

In [4]:
class Classifier:
    
    def __init__(self):
        """
        Initializes logistic regression classifier """

        self.model = LogisticRegression(
             max_iter = 2000,
             class_weight = "balanced",
             solver = "liblinear",
             C=0.5,
             random_state=42
        )
        self.is_trained = False

    def train(self, features, labels):
        self.model.fit(features, labels)
        self.is_trained = True

    def predict(self, features):
        return self.model.predict(features)

In [5]:
#Using pandas to handle the information

training_texts = train_df["TEXT"].fillna("")
training_labels = train_df["LABEL"]
test_texts = test_df["TEXT"].fillna("")

In [6]:
train_texts, dev_texts, train_labels, dev_labels = train_test_split(
    training_texts, training_labels,
    test_size=0.2,
    random_state=42,
    stratify=training_labels
)

In [7]:
#Debugging
print(f"Training set size: {len(train_texts)}")
print(f"Development set size: {len(dev_texts)}")
print(f"\nTraining label distribution:")
print(train_labels.value_counts().sort_index())
print(f"\nDevelopment label distribution:")
print(dev_labels.value_counts().sort_index())

Training set size: 56253
Development set size: 14064

Training label distribution:
LABEL
0    25831
1    15311
2    15111
Name: count, dtype: int64

Development label distribution:
LABEL
0    6458
1    3828
2    3778
Name: count, dtype: int64


In [8]:
#Features
to_features = TextToFeatures()
to_features.fit(train_texts)

#Labels
to_labels = TextToLabels()
to_labels.fit(train_labels)

In [9]:
clf = Classifier()
clf.train(to_features.transform(train_texts), to_labels.transform(train_labels))

dev_predictions = clf.predict(to_features.transform(dev_texts))
dev_labels_encoded = to_labels.transform(dev_labels)

accuracy = accuracy_score(dev_labels_encoded, dev_predictions)
f1 = f1_score(dev_labels_encoded, dev_predictions, average="weighted")

In [10]:
#Decoding
test_predictions = clf.predict(to_features.transform(test_texts))
test_predictions_decoded = to_labels.encoder.inverse_transform(test_predictions)

In [11]:
accuracy = accuracy_score(dev_labels_encoded, dev_predictions)
f1 = f1_score(dev_labels_encoded, dev_predictions, average="weighted")

print(f"\n📊 Development Set Results:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (weighted): {f1:.4f}")
print("\nClassification Report:")
print(classification_report(dev_labels_encoded, dev_predictions))


📊 Development Set Results:
Accuracy: 0.9123
F1 Score (weighted): 0.9119

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.97      6458
           1       0.86      0.85      0.85      3828
           2       0.87      0.85      0.86      3778

    accuracy                           0.91     14064
   macro avg       0.90      0.90      0.90     14064
weighted avg       0.91      0.91      0.91     14064



In [12]:
#But is it good?

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "LABEL": test_predictions_decoded
})

submission.to_csv("test.csv", index=False)